# Pelican animation on free Google Colab (T4 GPU)

This notebook clones [hosein135/animation1](https://github.com/hosein135/animation1) and renders `output/animation.mp4` on the **highest GPU available on free Colab**: an **NVIDIA Tesla T4** (16 GB).

**Before you run anything**

1. Open in Colab: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hosein135/animation1/blob/main/colab_gpu_render.ipynb)
2. **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (that is the top free-tier GPU; A100 / L4 are Colab Pro).
3. **Runtime → Run all**. Keep this tab open. The clip is **6 seconds at 12 fps, 1280×720** (about 72 frames) so Colab usually finishes in a few minutes. When render finishes, Colab **downloads `output/` as a zip**.

If a previous run failed: **re-run from the clone cell** (so Colab pulls the latest `scripts/`), then the install/render cells. Do not re-run only the render cell on a stale clone.

Free Colab is headless, so this run uses **Blender Cycles CUDA** on the T4 (not EEVEE). Samples and denoising are kept low for speed. Encode uses **NVENC** when FFmpeg supports it, otherwise libx264.


In [ ]:
# @title Config
REPO_URL = "https://github.com/hosein135/animation1.git"
BRANCH = "main"
REPO_DIR = "/content/animation1"

# Match run.ps1 / the official Linux build the pipeline expects.
BLENDER_VERSION = "4.5.5"
BLENDER_MAJOR = "4.5"
BLENDER_ROOT = "/usr/local/blender"

# Fast Colab path: Cycles CUDA, one worker. Timing/size come from scene.json (6s / 12fps / 720p).
ENGINE = "cycles"
WORKERS = 1
import os
os.environ["ANIM_CYCLES_DEVICE"] = "CUDA"
os.environ["ANIM_TEE_LOGS"] = "1"

In [ ]:
# Attach the T4 (or whichever NVIDIA GPU this VM actually gave you) and refuse CPU runtimes.
import os
import shutil
import subprocess

def require_colab_gpu():
    smi = shutil.which("nvidia-smi")
    if not smi:
        raise SystemExit(
            "No NVIDIA driver / nvidia-smi.\n"
            "Runtime → Change runtime type → Hardware accelerator → T4 GPU, then Run all again."
        )

    print(subprocess.check_output([smi, "-L"], text=True).strip())
    raw = subprocess.check_output(
        [
            smi,
            "--query-gpu=index,name,memory.total",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    )
    gpus = []
    for line in raw.strip().splitlines():
        idx_s, name, mem_s = [p.strip() for p in line.split(",", 2)]
        gpus.append((int(mem_s), int(idx_s), name))
    if not gpus:
        raise SystemExit("nvidia-smi listed no GPUs. Switch this runtime to T4 GPU.")

    # Highest GPU on this VM = most VRAM. Free Colab's top card is Tesla T4.
    gpus.sort(reverse=True)
    mem, idx, name = gpus[0]
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = str(idx)
    print(f"Using GPU {idx}: {name} ({mem} MiB)  [highest VRAM on this VM]")
    if "T4" not in name and "Tesla T4" not in name:
        print(
            "Note: free Colab's highest GPU is T4. "
            "If you see A100/L4 you are on Colab Pro; if you see nothing, switch runtime."
        )
    subprocess.check_call([smi])
    return name

gpu_name = require_colab_gpu()

In [ ]:
# Clone (or hard-reset to) the GitHub project.
import os
import shutil
import subprocess
from pathlib import Path

os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
repo = Path(REPO_DIR)
repo.parent.mkdir(parents=True, exist_ok=True)

def run(cmd, **kw):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, **kw)

if (repo / ".git").is_dir():
    run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", str(repo), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
    run(["git", "-C", str(repo), "reset", "--hard", f"origin/{BRANCH}"])
    run(["git", "-C", str(repo), "clean", "-fdx", "-e", "output"])
else:
    if repo.exists():
        shutil.rmtree(repo)
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo)])

os.chdir(repo)
print("Working directory:", Path.cwd())
print("HEAD:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

In [ ]:
# Install FFmpeg + Blender 4.5 Linux x64 (official build with CUDA/OptiX kernels).
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

def apt_install():
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(
        [
            "apt-get", "install", "-y", "-qq",
            "ffmpeg", "xz-utils", "curl",
            "libxrender1", "libxi6", "libxxf86vm1", "libxfixes3",
            "libsm6", "libxkbcommon0", "libgl1", "libegl1", "libx11-6",
        ]
    )

def blender_bin() -> Path:
    return Path(BLENDER_ROOT) / "blender"

def install_blender():
    exe = blender_bin()
    if exe.is_file():
        print("Blender already installed:", exe)
        return
    name = f"blender-{BLENDER_VERSION}-linux-x64"
    url = (
        f"https://download.blender.org/release/Blender{BLENDER_MAJOR}/"
        f"{name}.tar.xz"
    )
    tarball = Path("/content") / f"{name}.tar.xz"
    print("Downloading", url)
    subprocess.check_call(["curl", "-L", "--retry", "5", "--retry-all-errors", "-o", str(tarball), url])
    extract_parent = Path("/usr/local")
    print("Extracting to", extract_parent)
    extract_kw = {}
    if sys.version_info >= (3, 12):
        extract_kw["filter"] = "data"
    with tarfile.open(tarball, "r:xz") as tf:
        tf.extractall(extract_parent, **extract_kw)
    extracted = extract_parent / name
    if extracted.is_dir():
        if Path(BLENDER_ROOT).exists():
            shutil.rmtree(BLENDER_ROOT)
        extracted.rename(BLENDER_ROOT)
    tarball.unlink(missing_ok=True)
    if not exe.is_file():
        raise SystemExit(f"Blender binary missing after extract: {exe}")

apt_install()
install_blender()

# Persist for later cells (shell `export` does not stick).
os.environ["PATH"] = f"{BLENDER_ROOT}:" + os.environ.get("PATH", "")
os.environ["PROJECT_ROOT"] = str(Path(REPO_DIR).resolve())
os.environ["DATA_DIR"] = str((Path(REPO_DIR) / "data").resolve())
os.environ["OUTPUT_DIR"] = str((Path(REPO_DIR) / "output").resolve())
Path(os.environ["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)

link = Path("/usr/local/bin/blender")
if link.exists() or link.is_symlink():
    link.unlink()
link.symlink_to(blender_bin())
print("blender:", shutil.which("blender"))
print("ffmpeg: ", shutil.which("ffmpeg"))

In [ ]:
# Confirm tools and that Cycles can see the T4.
import os
import subprocess
from pathlib import Path

os.environ["PATH"] = f"{BLENDER_ROOT}:" + os.environ.get("PATH", "")

print("== blender --version ==")
subprocess.check_call(["blender", "--version"])
print("\n== ffmpeg ==")
subprocess.check_call(["ffmpeg", "-hide_banner", "-version"])

probe_py = Path("/tmp/cycles_gpu_probe.py")
probe_py.write_text(
    "import bpy\n"
    "prefs = bpy.context.preferences.addons['cycles'].preferences\n"
    "found = False\n"
    "for compute in ('OPTIX', 'CUDA'):\n"
    "    try:\n"
    "        prefs.compute_device_type = compute\n"
    "        prefs.get_devices()\n"
    "        print(f'{compute}:')\n"
    "        for d in prefs.devices:\n"
    "            print(f'  {d.name} ({d.type})')\n"
    "            if d.type in {compute, 'CUDA', 'OPTIX'}:\n"
    "                found = True\n"
    "    except Exception as exc:\n"
    "        print(compute, 'unavailable:', exc)\n"
    "if not found:\n"
    "    raise SystemExit('Cycles did not list an NVIDIA GPU. Is the runtime T4?')\n"
)
print("\n== Cycles GPU probe ==")
subprocess.check_call(
    ["blender", "--background", "--factory-startup", "--python", str(probe_py)]
)

In [ ]:
# Render + encode on the T4, with a live progress bar in this cell.
import json
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

from tqdm.auto import tqdm

os.chdir(REPO_DIR)
os.environ["PATH"] = f"{BLENDER_ROOT}:" + os.environ.get("PATH", "")
os.environ["PROJECT_ROOT"] = str(Path(REPO_DIR).resolve())
os.environ["DATA_DIR"] = str((Path(REPO_DIR) / "data").resolve())
os.environ["OUTPUT_DIR"] = str((Path(REPO_DIR) / "output").resolve())
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["ANIM_CYCLES_DEVICE"] = "CUDA"
os.environ["ANIM_PROGRESS"] = "live"
os.environ["ANIM_TEE_LOGS"] = "0"
runtime = Path("/tmp/runtime-colab")
runtime.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_RUNTIME_DIR", str(runtime))

scene = json.loads((Path(REPO_DIR) / "data" / "scene.json").read_text(encoding="utf-8"))
total = max(1, int(round(float(scene["fps"]) * float(scene["duration_seconds"]))))
frames_dir = Path(REPO_DIR) / "output" / "frames"
mp4 = Path(REPO_DIR) / "output" / "animation.mp4"

def count_frames() -> int:
    if not frames_dir.is_dir():
        return 0
    return sum(1 for p in frames_dir.glob("frame_*.jpg")) + sum(
        1 for p in frames_dir.glob("frame_*.png")
    )

cmd = [
    sys.executable, "-u", "scripts/pipeline.py",
    "--renderer", "blender",
    "--engine", ENGINE,
    "--workers", str(WORKERS),
]
print("cwd:", os.getcwd())
print("+", " ".join(cmd), flush=True)

proc = subprocess.Popen(
    cmd,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
log_tail: list[str] = []

def _pump() -> None:
    assert proc.stdout is not None
    for line in proc.stdout:
        log_tail.append(line)
        if len(log_tail) > 500:
            del log_tail[:250]

threading.Thread(target=_pump, daemon=True).start()

render_bar = tqdm(total=total, desc="Render frames", unit="frm", dynamic_ncols=True)
encode_bar = None
last = 0
try:
    while proc.poll() is None:
        n = min(count_frames(), total)
        if n > last:
            render_bar.update(n - last)
            last = n
        if last >= total and encode_bar is None:
            render_bar.close()
            encode_bar = tqdm(total=1, desc="Encode MP4", unit="file")
        if encode_bar is not None and encode_bar.n == 0 and mp4.is_file() and mp4.stat().st_size > 0:
            encode_bar.update(1)
        time.sleep(0.25)
    n = min(count_frames(), total)
    if n > last:
        render_bar.update(n - last)
finally:
    if not render_bar.disable:
        render_bar.close()
    if encode_bar is not None and encode_bar.n == 0 and mp4.is_file() and mp4.stat().st_size > 0:
        encode_bar.update(1)
    if encode_bar is not None:
        encode_bar.close()

code = proc.wait()
if code != 0:
    print("".join(log_tail[-80:]), flush=True)
    logs = Path(REPO_DIR) / "output" / "logs"
    if logs.is_dir():
        for p in sorted(logs.glob("*.log")):
            print(f"\n===== {p} =====", flush=True)
            print(p.read_text(encoding="utf-8", errors="replace")[-16000:])
    raise subprocess.CalledProcessError(code, cmd)

print("Output:", mp4, "size=", mp4.stat().st_size if mp4.is_file() else 0)

# Zip the whole output folder and start a browser download.
import zipfile
from datetime import datetime, timezone

out_dir = Path(REPO_DIR) / "output"
if not out_dir.is_dir():
    raise SystemExit(f"Missing {out_dir}")

members = [p for p in out_dir.rglob("*") if p.is_file()]
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
zip_path = Path("/content") / f"animation1-output-{stamp}.zip"
print(f"Zipping {len(members)} files from {out_dir} → {zip_path}", flush=True)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in tqdm(members, desc="Zip output", unit="file"):
        zf.write(p, arcname=str(Path("output") / p.relative_to(out_dir)))
print(f"Zip ready: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)", flush=True)

try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception as exc:
    print("Browser download unavailable:", exc)
    print("Zip is at", zip_path)

In [ ]:
# Play the clip (the zip of output/ already downloaded from the render cell).
from pathlib import Path
from IPython.display import Video, display

mp4 = Path(REPO_DIR) / "output" / "animation.mp4"
if not mp4.is_file():
    raise SystemExit(f"Missing {mp4}")

print(f"{mp4} ({mp4.stat().st_size / 1e6:.1f} MB)")
display(Video(str(mp4), embed=True, width=960))